# 🕵️‍♀️ Operation Nightfall: Digital Forensics with Pandas

**Clearance Level:** TOP SECRET / EYES ONLY  
**Case ID:** #DFIR-2024-X99  
**Incident:** Lateral Movement / Data Exfiltration  

## 📂 Case Briefing
**To:** Senior Forensic Analyst  
**From:** CISO, Global Finance Corp  
**Subject:** URGENT: Suspicious Activity on FIN-WS-01

We have detected anomalous outbound traffic from workstation `FIN-WS-01` (Finance Dept). The user, "Alice", claims she was only using Excel, but our SIEM shows connections to a known threat actor IP. 

**Your Mission:**
1.  **Triage:** Load the system logs and established a baseline.
2.  **Hunt:** Filter through the noise to find "Patient Zero" (the initial malicious process).
3.  **Decrypt:** The attacker is using obfuscation. Decode the command line to find their target.
4.  **Correlate:** Connect the process execution to the network traffic.
5.  **Visualize:** **[NEW]** Build a Threat Hunting Console to see the attack vectors.

**Tools:** Python, Pandas, Matplotlib, and your investigative intuition.

---

## 🛠️ Step 0: Lab Setup & Evidence Extraction
Before we look at the evidence, we need to extract the logs from the "image" (i.e., generate our dataset). 

**Analyst Note:** In the real world, you'd be parsing raw EVTX (Windows Event Logs) or PCAP files. Today, we have these pre-parsed into CSV format for you. Run the cell below to generate the evidence files.

In [ ]:
import pandas as pd
import numpy as np
import random
import base64
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact
from datetime import datetime, timedelta

# === SYNTHETIC DATA GENERATOR [DO NOT EDIT] ===
def generate_evidence():
    np.random.seed(42)  # For reproducible crimes
    n_rows = 2000
    
    # --- 1. Process List (The "Haystack") ---
    print("Generating Process Logs...")
    processes = ['svchost.exe', 'explorer.exe', 'chrome.exe', 'excel.exe', 'teams.exe', 'onedrive.exe']
    weights =   [0.3,           0.1,            0.3,          0.15,        0.1,         0.05]
    
    data = {
        'Timestamp': [datetime(2024, 5, 12, 8, 0, 0) + timedelta(seconds=x*15) for x in range(n_rows)],
        'Process_Name': np.random.choice(processes, n_rows, p=weights),
        'PID': np.random.randint(1000, 9999, n_rows),
        'User': ['SYSTEM' if p == 'svchost.exe' else 'Alice_Finance' for p in processes],
        'CommandLine': ['-' for _ in range(n_rows)],
        'Memory_MB': np.random.uniform(10, 500, n_rows).round(1)
    }
    process_df = pd.DataFrame(data)
    
    # > INJECT RED HERRING (Bob_Dev)
    # Bob is an admin running nmap at 2:00 PM
    process_df.loc[100, 'Process_Name'] = 'nmap.exe'
    process_df.loc[100, 'User'] = 'Bob_Dev'
    process_df.loc[100, 'PID'] = 9999 # Bob's PID
    process_df.loc[100, 'CommandLine'] = 'nmap -sV 192.168.1.0/24'
    
    # > INJECT MALWARE (The "Needle")
    secret_cmd = "ping -c 4 185.10.10.99" 
    encoded_cmd = base64.b64encode(secret_cmd.encode()).decode()
    
    process_df.loc[500, 'Process_Name'] = 'powershell.exe' 
    process_df.loc[500, 'User'] = 'Alice_Finance'
    process_df.loc[500, 'PID'] = 6660
    process_df.loc[500, 'CommandLine'] = f"-enc {encoded_cmd}"
    
    process_df.loc[1200, 'Process_Name'] = 'mimikatz.exe'
    process_df.loc[1200, 'User'] = 'SYSTEM'
    process_df.loc[1200, 'PID'] = 1337
    
    process_df.loc[500, 'Process_Name'] = 'PowerShell.exe ' # Evil trailing space
    
    # --- 2. Network Logs (The "Traffic") ---
    print("Generating Network Logs...")
    n_net = 500
    net_data = {
        'Time': [datetime(2024, 5, 12, 9, 0, 0) + timedelta(seconds=x*60) for x in range(n_net)],
        'Source_IP': '192.168.1.105', 
        'Destination_IP': np.random.choice(['8.8.8.8', '10.0.0.5', '172.16.254.1', '104.21.5.2'], n_net),
        'Destination_Port': np.random.choice([443, 80, 53, 445], n_net),
        'Bytes_Sent': np.random.randint(100, 5000, n_net),
        'Protocol': 'TCP'
    }
    net_df = pd.DataFrame(net_data)
    net_df['PID'] = np.random.choice(process_df['PID'], n_net)

    # > INJECT BOB'S NOISE (The Port Scan)
    # Looks like a solid block on the chart
    scan_time = datetime(2024, 5, 12, 14, 0, 0)
    scan_data = {
        'Time': [scan_time + timedelta(milliseconds=x*10) for x in range(100)],
        'Source_IP': '192.168.1.105',
        'Destination_IP': [f'192.168.1.{x}' for x in range(100)], 
        'Destination_Port': [80]*100,
        'Bytes_Sent': [64]*100,
        'Protocol': 'TCP',
        'PID': 9999
    }
    net_df = pd.concat([net_df, pd.DataFrame(scan_data)], ignore_index=True)
    
    # > INJECT C2 TRAFFIC
    c2_ip = '185.10.10.99'
    net_df.loc[100, 'Destination_IP'] = c2_ip
    net_df.loc[100, 'Destination_Port'] = 8080
    net_df.loc[100, 'Bytes_Sent'] = 1500000
    net_df.loc[100, 'PID'] = 6660
    
    net_df.loc[300, 'Destination_IP'] = c2_ip
    net_df.loc[300, 'Destination_Port'] = 4444
    net_df.loc[300, 'Bytes_Sent'] = 5000000
    net_df.loc[300, 'PID'] = 6660
    
    process_df.to_csv('process_log.csv', index=False)
    net_df.to_csv('network_log.csv', index=False)
    print("✅ Evidence Files Created: process_log.csv, network_log.csv")

generate_evidence()

---

## 🔬 Mission Phases 1-4 (Recap)
**Objective:** Rapidly get back to where we were (Triage -> Hunt -> Verify -> Correlate).

*Run the cells below to restore our findings.*

In [ ]:
# 1. Load Data
process_df = pd.read_csv('process_log.csv')
net_df = pd.read_csv('network_log.csv')

# 2. Clean Data
process_df['clean_name'] = process_df['Process_Name'].str.lower().str.strip()

# 3. Identify Malicious PID
patient_zero = process_df[process_df['clean_name'] == 'powershell.exe']
malicious_pid = 6660
print(f"Malicious PID Identified: {malicious_pid}")

# 4. Decode Payload (CTF)
encoded_string = "cGluZyAtYyA0IDE4NS4xMC4xMC45OQ=="
decoded_bytes = base64.b64decode(encoded_string)
print("Decoded Payload:", decoded_bytes.decode('utf-8'))

# 5. Merge Data
merged_df = pd.merge(net_df, process_df[['PID', 'Process_Name', 'User']], on='PID', how='left')

---

## 🖥️ Mission Phase 6: The Threat Hunting Console
**Objective:** Visualize the attack vectors using an Interactive Dashboard.

Numbers in a CSV are hard to read. A **scatter plot** of traffic can reveal patterns instantly.

We will compare:
*   **X-Axis:** Time
*   **Y-Axis:** Port / Bytes
*   **Color:** The Process Responsible

In [ ]:
# Pre-processing for Plotting
merged_df['Time'] = pd.to_datetime(merged_df['Time'])
merged_df['Hour'] = merged_df['Time'].dt.hour

@interact
def show_threat_dashboard(
    x_axis = ['Time', 'Bytes_Sent', 'Destination_Port'],
    y_axis = ['Destination_Port', 'Bytes_Sent', 'Hour'],
    color_by = ['Process_Name', 'Protocol']
):
    plt.figure(figsize=(12, 6))
    sns.scatterplot(data=merged_df, x=x_axis, y=y_axis, hue=color_by, palette='viridis', s=100, alpha=0.7)
    plt.title(f"Threat Vector: {x_axis} vs {y_axis}")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

### 🕵️‍♀️ Visual Analysis Tasks
Use the console above to answer these questions:

1.  **Set X='Time', Y='Destination_Port', Color='Process_Name':**
    *   Do you see the **Solid Line** of dots around 14:00? That is **Bob's Nmap Scan** (hitting every port in sequence).
    *   Do you see the lonely dots for `powershell.exe`? Note the unusual port usage.
2.  **Set X='Bytes_Sent', Y='Destination_Port':**
    *   Find the **Anomalies**. Most traffic is small (DNS/Web). The **Data Exfiltration** stands out as the massive dot on the far right (High Bytes).

---

## 🤖 Mission Phase 7: The AI Connection (From Pixels to Vectors)

You just used your eyes to spot the "Port Scan" (The solid line) and the "Exfiltration" (The big dot). 

**This is exactly how AI cybersecurity works.**

### 1. Computer Vision (CNN)
If we took a Screenshot of that scatter plot and fed it into a **Convolutional Neural Network (CNN)**—the same AI that recognizes cats in photos—it would recognize the **Shape of an Attack**.
*   **Vertical Line** = Port Scan
*   **Dense Cluster** = DDoS Attack

### 2. Sequence Learning (RNN/LSTM)
Instead of an image, if we feed the **Sequence of Vectors** (`[Time, IP, Port, Bytes]`) into a **Recurrent Neural Network (RNN)**, the AI learns the "rhythm" of normal traffic.
*   **Normal:** `[Http, Http, Dns, Http]`
*   **Anomaly:** `[Port 1, Port 2, Port 3...]` -> **Alert!**

> **Future Learning:** In advanced courses, you will take this same DataFrame and train a Random Forest classifier to detect `powershell.exe` automatically!

## 🏆 Mission Complete

You have gone from basic Pandas filters to building an AI-ready Threat Dashboard. 

**Final Status:** Report Submitted. Threat Neutralized.